## 1. Cek GPU dan pasang library

In [1]:
!nvidia-smi
!pip install -q ultralytics==8.4.193

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

def cari(nama):
    for p in (f'/content/{nama}', f'/content/drive/MyDrive/acne_v2/{nama}', f'/content/drive/MyDrive/{nama}'):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(nama + ' tidak ditemukan. Upload ke MyDrive/acne_v2/')

zip_lama, zip_baru, model_lama = cari('deteksi.zip'), cari('Acne new data.v1i.yolov8.zip'), cari('deteksi_yolov8n_50ep.pt')
print(zip_lama, zip_baru, model_lama, sep='\n')
!unzip -q -o "{zip_lama}" -d /content
!unzip -q -o "{zip_baru}" -d /content/acne_new
!ls /content/deteksi /content/acne_new

## 3. Pilih foto Blackheads dan buat dataset
Langkah:
1. **Pilih kelas.** Dari 15 kelas "Acne new data":
   - dijadikan `Acne`: Acne, Blackhead, Conglobata, Cystic, Papular, Purulent, Whitehead
   - bukan jerawat, kotaknya dihapus (model belajar ini bukan jerawat): Flat_wart, Keloid, Scars, Syringoma
   - mirip jerawat tapi bukan, atau tidak jelas: Crystanlline, Sebo-crystan-conglo, Folliculitis, Milium.
     **Foto yang berisi kelas ini dibuang**, supaya model tidak bingung.
2. **Hanya foto yang punya kotak Blackhead** yang dipakai. Foto lain dibuang (`tanpa_blackhead`).
3. **Isi jerawat yang lupa dikotak.** Model 1 lama menebak tiap foto. Tebakan yakin (>= `CONF_ISI`) yang tidak bertumpuk
   dengan kotak label mana pun ditambahkan sebagai kotak `Acne`, **hanya di train**.
   Test baru tetap memakai label asli, supaya penilaian tidak menguntungkan model lama.
4. **Cegah bocoran.** Foto yang mirip foto valid/test lama (embedding ResNet50 >= 0,95) dibuang; yang mirip foto train lama dipaksa masuk train.
5. **Bagi per foto asli** (nama sama atau foto sangat mirip = satu kelompok) 80% train, 20% test.

Hasil di `/content/deteksi_bh`. Training memakai train lama + train baru. Validasi (untuk memilih `best.pt`)
**hanya valid lama**, karena labelnya rapi; jadi tidak ada valid baru. Test lama (48 foto) **tidak diubah**. Test baru (`deteksi_bh/test`) dipakai
untuk mengukur Blackheads.

Butuh sekitar 10 menit.

In [2]:
import glob, json, os, random, re, shutil
from collections import Counter, defaultdict

import numpy as np
import torch
import torchvision
from PIL import Image
from ultralytics import YOLO

# ---------- pengaturan ----------
LAMA = '/content/deteksi'
SUMBER = '/content/acne_new'
TUJUAN = '/content/deteksi_bh'
NAMA_KELAS = ['Acne', 'Blackhead', 'Conglobata', 'Crystanlline', 'Cystic', 'Flat_wart', 'Folliculitis', 'Keloid',
              'Milium', 'Papular', 'Purulent', 'Scars', 'Sebo-crystan-conglo', 'Syringoma', 'Whitehead']
JERAWAT = {'Acne', 'Blackhead', 'Conglobata', 'Cystic', 'Papular', 'Purulent', 'Whitehead'}
BUANG_FOTO = {'Crystanlline', 'Sebo-crystan-conglo', 'Folliculitis', 'Milium'}
CONF_ISI = 0.4            # tebakan model lama seyakin ini dipakai mengisi jerawat yang lupa dikotak
AMBANG_FOTO = 0.95       # dua foto semirip ini dianggap foto asli yang sama
PORSI = {'train': 0.80, 'test': 0.20}   # tanpa valid baru: best.pt dipilih dengan valid lama yang labelnya rapi
SEED = 42
rng = random.Random(SEED)


def nama_asli(stem):
    return re.sub(r'_(jpg|jpeg|png)\.rf\.[0-9a-f]+$', '', stem, flags=re.I).lower()


def ke_xyxy(xc, yc, w, h, W, H):
    return ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)


def iou(a, b):
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    irisan = max(0, x1 - x0) * max(0, y1 - y0)
    return irisan / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - irisan + 1e-9)


def cocok(tebak, label):
    """Tebakan dianggap mengenai kotak label kalau IoU >= 0.2 atau titik tengahnya di dalam kotak label."""
    cx, cy = (tebak[0] + tebak[2]) / 2, (tebak[1] + tebak[3]) / 2
    return iou(tebak, label) >= 0.2 or (label[0] <= cx <= label[2] and label[1] <= cy <= label[3])


# 1. kumpulkan foto dan kotak
gambar = {}
for p in glob.glob(f'{SUMBER}/*/images/*'):
    gambar[(p.split('/')[-3], os.path.splitext(os.path.basename(p))[0])] = p
foto = []
for lbl in sorted(glob.glob(f'{SUMBER}/*/labels/*.txt')):
    sp, stem = lbl.split('/')[-3], os.path.splitext(os.path.basename(lbl))[0]
    if (sp, stem) not in gambar:
        continue
    kotak = []
    for t in open(lbl).read().split('\n'):
        p = t.split()
        if len(p) == 5:
            kotak.append((NAMA_KELAS[int(float(p[0]))], *map(float, p[1:])))
    nama = {k[0] for k in kotak}
    buang = 'kelas_ragu' if nama & BUANG_FOTO else ('' if 'Blackhead' in nama else 'tanpa_blackhead')
    foto.append({'gambar': gambar[(sp, stem)], 'stem': stem, 'kotak': kotak, 'buang': buang})
print('Foto "Acne new data":', len(foto))

# 2. jerawat yang lupa dikotak: tebakan yakin model lama yang tidak mengenai kotak label mana pun
pemeriksa = YOLO(model_lama)
calon = [f for f in foto if not f['buang']]
for i in range(0, len(calon), 64):
    kelompok = calon[i:i + 64]
    for f, r in zip(kelompok, pemeriksa.predict([f['gambar'] for f in kelompok], imgsz=640, conf=CONF_ISI,
                                                iou=0.45, verbose=False)):
        H, W = r.orig_shape
        semua_label = [ke_xyxy(*k[1:], W, H) for k in f['kotak']]   # termasuk bekas luka dll: bukan "lupa dikotak"
        f['ukuran'] = (W, H)
        f['tanpa_kotak'] = [b for b in r.boxes.xyxy.tolist() if not any(cocok(b, l) for l in semua_label)]
print(f'  diperiksa {len(calon)} foto berisi Blackhead', flush=True)

# 3. sidik jari foto utuh untuk bocoran dan kelompok
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
bobot = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
resnet = torchvision.models.resnet50(weights=bobot)
resnet.fc = torch.nn.Identity()
resnet = resnet.eval().to(dev)
tf = bobot.transforms()


def sidik_jari(paths, nama, batch=128):
    hasil = []
    with torch.no_grad():
        for i in range(0, len(paths), batch):
            x = torch.stack([tf(Image.open(p).convert('RGB')) for p in paths[i:i + batch]]).to(dev)
            e = resnet(x) + resnet(torch.flip(x, dims=[3]))
            hasil.append(torch.nn.functional.normalize(e, dim=1).cpu().numpy())
    print(f'  sidik jari {nama}: {len(paths)}', flush=True)
    return np.concatenate(hasil) if hasil else np.zeros((0, 2048), np.float32)


def maks_mirip(A, B, blok=2048):
    if not len(A) or not len(B):
        return np.zeros(len(A))
    return np.concatenate([(A[a:a + blok] @ B.T).max(1) for a in range(0, len(A), blok)])


lolos = [f for f in foto if not f['buang']]
EB = sidik_jari([f['gambar'] for f in lolos], 'foto lolos')
E_uji_lama = sidik_jari(sorted(glob.glob(f'{LAMA}/valid/images/*') + glob.glob(f'{LAMA}/test/images/*')), 'valid/test lama')
E_train_lama = sidik_jari(sorted(glob.glob(f'{LAMA}/train/images/*')), 'train lama')
for f, a, b in zip(lolos, maks_mirip(EB, E_uji_lama), maks_mirip(EB, E_train_lama)):
    if a >= AMBANG_FOTO:
        f['buang'] = 'mirip_uji_lama'
    f['wajib_train'] = b >= AMBANG_FOTO

# 4. kelompok per foto asli, lalu bagi
induk = list(range(len(lolos)))
def cari_induk(a):
    while induk[a] != a:
        induk[a] = induk[induk[a]]
        a = induk[a]
    return a
pertama = {}
for i, f in enumerate(lolos):
    k = nama_asli(f['stem'])
    if k in pertama:
        induk[cari_induk(i)] = cari_induk(pertama[k])
    else:
        pertama[k] = i
for a, b in zip(*np.nonzero(np.triu(EB @ EB.T, 1) >= AMBANG_FOTO)):
    induk[cari_induk(int(a))] = cari_induk(int(b))
grup = defaultdict(list)
for i, f in enumerate(lolos):
    if not f['buang']:
        grup[cari_induk(i)].append(f)
daftar = list(grup.values())
rng.shuffle(daftar)
daftar.sort(key=len, reverse=True)
n_bh = lambda g: sum(k[0] == 'Blackhead' for f in g for k in f['kotak'])
total = {'foto': sum(len(g) for g in daftar), 'bh': sum(n_bh(g) for g in daftar)}
isi = {s: {'foto': 0, 'bh': 0} for s in PORSI}
for g in daftar:
    if any(f['wajib_train'] for f in g):
        s = 'train'
    else:
        c = {'foto': len(g), 'bh': n_bh(g)}
        s = max(PORSI, key=lambda s: sum(c[k] * (PORSI[s] * total[k] - isi[s][k]) / max(1, total[k]) for k in c))
    isi[s]['foto'] += len(g)
    isi[s]['bh'] += n_bh(g)
    for f in g:
        f['split'] = s

# 5. tulis dataset baru (label 1 kelas). Baris label: kotak label asli dulu, lalu kotak isian model lama.
if os.path.exists(TUJUAN):
    shutil.rmtree(TUJUAN)
jenis_test = {}
for f in foto:
    if f['buang']:
        continue
    s, nama = f['split'], 'B_' + os.path.basename(f['gambar'])
    os.makedirs(f'{TUJUAN}/{s}/images', exist_ok=True)
    os.makedirs(f'{TUJUAN}/{s}/labels', exist_ok=True)
    shutil.copy2(f['gambar'], f'{TUJUAN}/{s}/images/{nama}')
    jerawat = [k for k in f['kotak'] if k[0] in JERAWAT]
    baris = ['0 %.6f %.6f %.6f %.6f' % tuple(k[1:]) for k in jerawat]
    if s != 'test':
        W, H = f['ukuran']
        for x0, y0, x1, y1 in f['tanpa_kotak']:
            x0, y0, x1, y1 = max(0, x0), max(0, y0), min(W, x1), min(H, y1)
            baris.append('0 %.6f %.6f %.6f %.6f' % ((x0 + x1) / 2 / W, (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H))
    with open(f'{TUJUAN}/{s}/labels/{os.path.splitext(nama)[0]}.txt', 'w') as out:
        out.write('\n'.join(baris))
    if s == 'test':
        jenis_test[nama] = [list(k) for k in jerawat]
json.dump(jenis_test, open(f'{TUJUAN}/jenis_asli_test.json', 'w'))


def tulis_yaml(path, train, val):
    baris = lambda xs: '\n'.join(f'  - {x}' for x in xs)
    open(path, 'w').write(f"train:\n{baris(train)}\nval:\n{baris(val)}\nnc: 1\nnames: ['Acne']\n")


tulis_yaml('/content/data_bh.yaml', [f'{LAMA}/train/images', f'{TUJUAN}/train/images'], [f'{LAMA}/valid/images'])
tulis_yaml('/content/uji_lama.yaml', [f'{LAMA}/train/images'], [f'{LAMA}/test/images'])
tulis_yaml('/content/uji_baru.yaml', [f'{TUJUAN}/train/images'], [f'{TUJUAN}/test/images'])

alasan = Counter(f['buang'] or 'dipakai' for f in foto)
print('\nKeputusan per foto:', dict(alasan))
print(f"\n{'':8}{'foto baru':>10}{'kotak label':>13}{'kotak Blackhead':>17}{'kotak isian':>13}{'foto lama':>11}")
for s in ('train', 'valid', 'test'):
    fs = [f for f in foto if not f['buang'] and f['split'] == s]
    nj = sum(k[0] in JERAWAT for f in fs for k in f['kotak'])
    nb = sum(k[0] == 'Blackhead' for f in fs for k in f['kotak'])
    ni = 0 if s == 'test' else sum(len(f['tanpa_kotak']) for f in fs)
    print(f'{s:8}{len(fs):>10}{nj:>13}{nb:>17}{ni:>13}{len(glob.glob(f"{LAMA}/{s}/images/*")):>11}')
print('(test baru tanpa kotak isian, supaya penilaian adil)')

Foto "Acne new data": 0


NameError: name 'model_lama' is not defined

### 3b. Cek hasil dengan mata (boleh dilewati)
10 foto train acak. Hijau = kotak label jerawat, kuning = kotak label Blackhead,
**biru = kotak isian dari model lama** (jerawat yang lupa dikotak).

Urutan setelah ini: **3d → 3e → 3c**.

In [ ]:
import matplotlib.pyplot as plt
from PIL import ImageDraw


def gambar_cek(f):
    im = Image.open(f['gambar']).convert('RGB')
    W, H = im.size
    dr = ImageDraw.Draw(im)
    for k in f['kotak']:
        if k[0] in JERAWAT:
            dr.rectangle(ke_xyxy(*k[1:], W, H), outline=(255, 220, 0) if k[0] == 'Blackhead' else (0, 220, 0), width=2)
    for b in f.get('tanpa_kotak', []):
        dr.rectangle(b, outline=(0, 120, 255), width=3)
    return im


r = random.Random(3)
dipakai = [f for f in foto if not f['buang'] and f['split'] == 'train']
fig, ax = plt.subplots(2, 5, figsize=(20, 8.5))
for i, f in enumerate(r.sample(dipakai, min(10, len(dipakai)))):
    ax[i // 5, i % 5].imshow(gambar_cek(f))
    ax[i // 5, i % 5].set_title(f"isian {len(f['tanpa_kotak'])} kotak", fontsize=10)
for a in ax.flat:
    a.axis('off')
plt.tight_layout()
plt.show()

### 3d. Lembar pemeriksaan label Blackhead
Sebagian label Blackhead di "Acne new data" ternyata **tahi lalat** atau kulit polos. Sel ini menampilkan semua potongan
Blackhead yang dipakai (train dan test), masing-masing **bernomor**, dengan kotak label berwarna kuning.
Lembar juga disimpan ke Drive `MyDrive/acne_v2/cek_blackhead/`, jadi bisa dibagi ke anggota tim.

Model 1 hanya punya 1 kelas ("Acne"), jadi **yang ditandai hanya yang BUKAN jerawat**:
- ❌ **tahi lalat**: cokelat atau hitam, bulat, warnanya rata, sering menonjol dan lebih besar dari pori
- ❌ **tidak ada apa-apa** di dalam kotak (kulit polos)
- ❌ flek, bekas luka, rambut, bayangan
- ✅ **biarkan**: titik hitam kecil di pori (Blackheads), bintik putih, benjolan merah (tetap jerawat)
- **Ragu? Biarkan.** Tandai hanya yang yakin bukan jerawat.

Tulis nomornya di sel 3e. Satu jerawat bisa muncul beberapa kali (salinan foto yang dibalik atau diputar); tandai semuanya.
Kalau sel 3 dijalankan ulang, jalankan sel ini lagi, karena nomornya bisa berubah.

In [ ]:
import json, math
from IPython.display import display
from PIL import ImageDraw, ImageFont

LEMBAR = '/content/drive/MyDrive/acne_v2/cek_blackhead'
KOLOM, BARIS, T = 10, 6, 120

daftar_bh = []
for f in sorted((f for f in foto if not f['buang']), key=lambda f: (f['split'], f['gambar'])):
    jerawat = [k for k in f['kotak'] if k[0] in JERAWAT]
    nama = 'B_' + os.path.basename(f['gambar'])
    for j, k in enumerate(jerawat):
        if k[0] == 'Blackhead':
            daftar_bh.append({'no': len(daftar_bh), 'split': f['split'], 'label': os.path.splitext(nama)[0] + '.txt',
                              'gambar_test': nama, 'baris': j, 'sumber': f['gambar'], 'kotak': list(k[1:])})
json.dump(daftar_bh, open(f'{TUJUAN}/daftar_blackhead.json', 'w'))


def potongan_bh(d):
    im = Image.open(d['sumber']).convert('RGB')
    W, H = im.size
    x0, y0, x1, y1 = ke_xyxy(*d['kotak'], W, H)
    r = max(x1 - x0, y1 - y0, 40) * 0.9
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    p = im.crop((cx - r, cy - r, cx + r, cy + r))
    ImageDraw.Draw(p).rectangle((x0 - cx + r, y0 - cy + r, x1 - cx + r, y1 - cy + r), outline=(255, 220, 0), width=2)
    return p.resize((T, T))


os.makedirs(LEMBAR, exist_ok=True)
font = ImageFont.load_default(size=16)
per_lembar = KOLOM * BARIS
n_lembar = math.ceil(len(daftar_bh) / per_lembar)
for l in range(n_lembar):
    isi_lembar = daftar_bh[l * per_lembar:(l + 1) * per_lembar]
    out = Image.new('RGB', (KOLOM * T, BARIS * (T + 4)), 'white')
    dr = ImageDraw.Draw(out)
    for i, d in enumerate(isi_lembar):
        x, y = (i % KOLOM) * T, (i // KOLOM) * (T + 4)
        out.paste(potongan_bh(d), (x, y))
        dr.rectangle((x, y, x + 44, y + 19), fill='white')
        dr.text((x + 3, y + 1), str(d['no']), fill='black', font=font)
    out.save(f'{LEMBAR}/lembar_{l + 1:02d}.jpg', quality=90)
    print(f"Lembar {l + 1}/{n_lembar}: nomor {isi_lembar[0]['no']} sampai {isi_lembar[-1]['no']}")
    display(out)
print('Total potongan Blackhead:', len(daftar_bh), dict(Counter(d['split'] for d in daftar_bh)))
print('Lembar disimpan di', LEMBAR)

### 3e. Hapus kotak yang bukan jerawat
Tulis nomor dari sel 3d di `SALAH`, dipisah koma. Rentang boleh pakai tanda minus. Contoh: `SALAH = '3, 17, 40-45, 120'`.
Kotak itu dihapus dari label, sehingga model belajar bahwa benda itu **bukan jerawat**.

Sel ini boleh dijalankan berkali-kali: label selalu dibangun ulang dari salinan asli (`labels_asli`),
jadi kalau ada nomor yang salah ketik, cukup perbaiki lalu jalankan lagi.
Daftar nomor juga disimpan ke Drive (`cek_blackhead/nomor_salah.txt`) untuk dokumentasi.

In [ ]:
SALAH = ''   # contoh: '3, 17, 40-45, 120'

import json, os, shutil
from collections import Counter, defaultdict


def baca_nomor(teks):
    hasil = set()
    for bagian in teks.replace(' ', '').replace('\n', '').split(','):
        if '-' in bagian:
            a, b = map(int, bagian.split('-'))
            hasil.update(range(a, b + 1))
        elif bagian:
            hasil.add(int(bagian))
    return hasil


daftar_bh = json.load(open(f'{TUJUAN}/daftar_blackhead.json'))
salah = baca_nomor(SALAH)
luar = sorted(n for n in salah if not 0 <= n < len(daftar_bh))
assert not luar, f'Nomor di luar daftar (0-{len(daftar_bh) - 1}): {luar}'

hapus = defaultdict(set)
for n in salah:
    d = daftar_bh[n]
    hapus[(d['split'], d['label'])].add(d['baris'])

for s in ('train', 'test'):
    asli = f'{TUJUAN}/{s}/labels_asli'
    if not os.path.exists(asli):
        shutil.copytree(f'{TUJUAN}/{s}/labels', asli)
    for nama in os.listdir(asli):
        baris = open(f'{asli}/{nama}').read().split('\n')
        buang = hapus.get((s, nama), set())
        open(f'{TUJUAN}/{s}/labels/{nama}', 'w').write('\n'.join(b for i, b in enumerate(baris) if i not in buang))
    for c in glob.glob(f'{TUJUAN}/{s}/*.cache'):
        os.remove(c)

asli_json = f'{TUJUAN}/jenis_asli_test_asli.json'
if not os.path.exists(asli_json):
    shutil.copy2(f'{TUJUAN}/jenis_asli_test.json', asli_json)
jenis = json.load(open(asli_json))
for d in daftar_bh:
    if d['split'] == 'test' and d['baris'] in hapus.get(('test', d['label']), set()):
        jenis[d['gambar_test']][d['baris']] = None
json.dump({k: [x for x in v if x] for k, v in jenis.items()}, open(f'{TUJUAN}/jenis_asli_test.json', 'w'))

LEMBAR = '/content/drive/MyDrive/acne_v2/cek_blackhead'
open(f'{LEMBAR}/nomor_salah.txt', 'w').write(SALAH)
json.dump([daftar_bh[n] for n in sorted(salah)], open(f'{TUJUAN}/blackhead_dihapus.json', 'w'), indent=1)
per = Counter(daftar_bh[n]['split'] for n in salah)
total = Counter(d['split'] for d in daftar_bh)
print(f'Kotak Blackhead dihapus: {len(salah)} dari {len(daftar_bh)} ({len(salah) / max(1, len(daftar_bh)):.0%})')
for s in ('train', 'test'):
    print(f'  {s:6} dihapus {per[s]:>4} | tersisa {total[s] - per[s]:>5}')

### 3c. Simpan dataset baru ke Drive
Jalankan **setelah** sel 3e, supaya label yang sudah dibersihkan ikut tersimpan.
Kalau sel 3e dijalankan lagi setelah ini, jalankan sel ini lagi juga.

In [ ]:
!cd /content && rm -f /content/drive/MyDrive/acne_v2/deteksi_bh.zip && zip -q -r /content/drive/MyDrive/acne_v2/deteksi_bh.zip deteksi_bh
!ls -lh /content/drive/MyDrive/acne_v2/

## 4. Training (les tambahan)
- **Mulai dari model 1 lama**, bukan dari nol, jadi yang sudah dikuasai tidak dibuang.
- Learning rate kecil (`lr0=0.002`, SGD) supaya perubahannya pelan dan tidak merusak.
- `imgsz=640`, sama dengan model lama dan ukuran foto aslinya.
- 25 epoch, berhenti lebih awal kalau 10 epoch tidak ada kemajuan.

Angka mAP50 di layar dihitung pada **valid lama** (56 foto), jadi bisa langsung dibandingkan dengan model lama
(0,636 di 640). Angkanya **seharusnya tidak jatuh jauh** di epoch awal; kalau jatuh di bawah 0,4 dan tidak naik lagi, hentikan.
Perkiraan **20-30 menit**. Biarkan tab Colab terbuka.

In [ ]:
from ultralytics import YOLO

model = YOLO(model_lama)
model.train(
    data='/content/data_bh.yaml',
    epochs=25,
    imgsz=640,
    batch=32,
    device=0,
    optimizer='SGD',
    lr0=0.002,
    lrf=0.1,
    warmup_epochs=1,
    patience=10,
    seed=0,
    project='/content/drive/MyDrive/acne_det_runs',
    name='deteksi_v3_bh_640',
)

## 4c. Training model lebih besar: YOLOv8s
`yolov8s` sekitar 3x lebih besar dari `yolov8n`, jadi kemungkinan lebih pintar, tapi aplikasi sedikit lebih lambat.
Karena belum pernah belajar jerawat, dia **dilatih dari awal** (bobot umum COCO) dengan data yang sama dengan sel 4:
data lama + foto Blackheads. Pengaturan seperti model lama: 50 epoch, `imgsz=640`, `patience=15`.

Perkiraan **45-70 menit**. Sel 4 dan 4c tidak saling bergantung; kalau waktu terbatas, boleh jalankan salah satu saja.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
model.train(
    data='/content/data_bh.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    patience=15,
    seed=0,
    project='/content/drive/MyDrive/acne_det_runs',
    name='deteksi_v3_bh_v8s_640',
)

## 4b. HANYA kalau training TERPUTUS di tengah jalan
**Jangan jalankan sel ini kalau sel 4 selesai normal.** Kalau terputus: jalankan sel 1, 2, unzip dataset dari Drive
(`!unzip -q -o /content/drive/MyDrive/acne_v2/deteksi_bh.zip -d /content`), lalu sel ini.
Sel 5 juga butuh file yaml dari sel 3; kalau sel 3 tidak dijalankan ulang, jalankan dulu bagian `tulis_yaml` di sel 3.

In [ ]:
import glob, os
from ultralytics import YOLO
last = max(glob.glob('/content/drive/MyDrive/acne_det_runs/deteksi_v3_bh_*640*/weights/last.pt'), key=os.path.getmtime)
print('Lanjut dari:', last)
YOLO(last).train(resume=True)

## 5. Bandingkan model lama dan baru
Peserta (yang belum dilatih otomatis dilewati):
- `lama@640`: model lama seperti di aplikasi sekarang
- `v3n@640`: YOLOv8n hasil les tambahan (sel 4)
- `v3s@640`: YOLOv8s model lebih besar (sel 4c)
- `v3n@960`: percobaan lama `deteksi_v3_bh_960` di Drive, sebagai pembanding

Ujian:
- **test lama** (48 foto, sama dengan ujian model lama): mAP50, precision, recall. **Tidak boleh turun.**
- **test baru** (foto Blackheads, label asli yang sudah dibersihkan): mAP50, precision, recall, dan
  **recall per jenis**, yaitu berapa persen jerawat tiap jenis yang dikotak. Baris Blackhead yang paling penting.

Precision di test baru bisa terlihat lebih rendah dari kenyataan: kalau masih ada jerawat yang tidak dikotak label,
kotak benar dari model dihitung salah. Recall lebih bisa dipercaya.

In [ ]:
import glob, json, os
from collections import Counter
from ultralytics import YOLO

TUJUAN = '/content/deteksi_bh'


def ke_xyxy(xc, yc, w, h, W, H):
    return ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)


def iou(a, b):
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    irisan = max(0, x1 - x0) * max(0, y1 - y0)
    return irisan / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - irisan + 1e-9)


def cocok(tebak, label):
    cx, cy = (tebak[0] + tebak[2]) / 2, (tebak[1] + tebak[3]) / 2
    return iou(tebak, label) >= 0.2 or (label[0] <= cx <= label[2] and label[1] <= cy <= label[3])


def recall_jenis(model, imgsz, conf=0.25):
    jenis = json.load(open(f'{TUJUAN}/jenis_asli_test.json'))
    nama = sorted(jenis)
    ketemu, total = Counter(), Counter()
    for i in range(0, len(nama), 32):
        hasil = model.predict([f'{TUJUAN}/test/images/{n}' for n in nama[i:i + 32]], imgsz=imgsz, conf=conf, iou=0.45, verbose=False)
        for n, r in zip(nama[i:i + 32], hasil):
            H, W = r.orig_shape
            tebak = r.boxes.xyxy.tolist()
            for k, *b in jenis[n]:
                total[k] += 1
                ketemu[k] += any(cocok(t, ke_xyxy(*b, W, H)) for t in tebak)
    return {k: (ketemu[k], total[k]) for k in sorted(total)}


RUNS = '/content/drive/MyDrive/acne_det_runs'
model_lama = '/content/drive/MyDrive/acne_v2/deteksi_yolov8n_50ep.pt'
peserta = [('lama@640', model_lama, 640)]
for nama, pola, ukuran in (('v3n@640', 'deteksi_v3_bh_640*', 640), ('v3s@640', 'deteksi_v3_bh_v8s_640*', 640),
                           ('v3n@960', 'deteksi_v3_bh_960', 960)):
    ada = glob.glob(f'{RUNS}/{pola}/weights/best.pt')
    if ada:
        peserta.append((nama, max(ada, key=os.path.getmtime), ukuran))
for p in peserta:
    print(p[0], p[1])
path_baru = peserta[-1][1] if len(peserta) > 1 else None
tabel, rj = [], {}
for nama, path, ukuran in peserta:
    m = YOLO(path)
    for ujian in ('uji_lama', 'uji_baru'):
        r = m.val(data=f'/content/{ujian}.yaml', split='val', imgsz=ukuran, batch=16, plots=False, verbose=False)
        tabel.append((nama, ujian.replace('uji_', 'test '), r.box.map50, r.box.mp, r.box.mr))
    rj[nama] = recall_jenis(m, ukuran)

teks = f"{'model':10}{'ujian':12}{'mAP50':>8}{'P':>8}{'R':>8}\n"
for nama, ujian, m50, p, r in sorted(tabel, key=lambda t: t[1]):
    teks += f'{nama:10}{ujian:12}{m50:>8.3f}{p:>8.3f}{r:>8.3f}\n'
teks += f"\nRecall per jenis di test baru (conf 0.25)\n{'jenis':12}{'jumlah':>8}" + ''.join(f'{p[0]:>11}' for p in peserta) + '\n'
for k in rj['lama@640']:
    n = rj['lama@640'][k][1]
    teks += f'{k:12}{n:>8}' + ''.join(f'{rj[p[0]][k][0] / n:>11.1%}' for p in peserta) + '\n'
print(teks)
open(f'{RUNS}/laporan_v3.txt', 'w').write(teks)
print('Disimpan ke', f'{RUNS}/laporan_v3.txt')

## 6. Unduh model

In [ ]:
import shutil
from google.colab import files

PILIH = 'v3s@640'   # ganti dengan nama peserta terbaik di tabel sel 5
asal = dict((p[0], p[1]) for p in peserta)[PILIH]
nama_file = {'v3n@640': 'deteksi_yolov8n_v3_bh_640.pt', 'v3s@640': 'deteksi_yolov8s_v3_bh_640.pt',
             'v3n@960': 'deteksi_yolov8n_v3_bh_960.pt'}[PILIH]
shutil.copy2(asal, f'/content/{nama_file}')
files.download(f'/content/{nama_file}')

Unduh hanya kalau sel 5 menunjukkan model baru lolos (test lama tidak turun, Blackheads naik).
File sudah diberi nama baru, jadi model lama tidak tertimpa.
Kirim tabel sel 3 dan sel 5 ke Claude.